In [8]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Layer
from tensorflow.keras.models import Model
from transformers import LongformerTokenizer, TFLongformerForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import backend as K

# Setting a global random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Define text cleaning function
def nltk_clean_text(text):
    text = str(text) if not pd.isna(text) else ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    tokens = word_tokenize(text)
    tokens = [w.lower() for w in tokens if w.isalpha()]
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(lemmatized)

# Load and preprocess the dataset
df = pd.read_csv(r"C:\Users\jpers\Desktop\NLP\USE THIS FOR NLP 2.csv")
df['rating_category'] = df['rating'].apply(lambda x: 'positive' if int(x) > 5 else 'neutral' if int(x) == 5 else 'negative')
df['review'] = df['review'].apply(nltk_clean_text)
df['Extracted Information'] = df['Extracted Information'].fillna("").apply(nltk_clean_text)
df['condition'] = df['condition'].apply(nltk_clean_text)
df['drug_name_x'] = df['drug_name_x'].apply(nltk_clean_text)
df['rating_norm'] = (df['rating'] - df['rating'].min()) / (df['rating'].max() - df['rating'].min())  # Normalize ratings
df['combined_text'] = df['review'] + " " + df['condition'] + " " + df['rating_category'] 


# Encode the rating category
label_encoder = LabelEncoder()
df['rating_category_encoded'] = label_encoder.fit_transform(df['rating_category'])
labels = tf.keras.utils.to_categorical(df['rating_category_encoded'], num_classes=3)

# Initialize tokenizer
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

# Function to encode reviews
def encode_reviews(reviews, tokenizer):
    return tokenizer.batch_encode_plus(
        reviews.tolist(),
        padding='max_length',
        truncation=True,
        max_length=1024,
        return_tensors='tf'
    )

# Encode data
X_train, X_test, y_train, y_test = train_test_split(df[['combined_text']], labels, test_size=0.2, random_state=42)
X_train_enc = encode_reviews(X_train['combined_text'], tokenizer)
X_test_enc = encode_reviews(X_test['combined_text'], tokenizer)

# Define the model
input_ids = Input(shape=(1024,), dtype='int32', name='input_ids')
attention_mask = Input(shape=(1024,), dtype='int32', name='attention_mask')

# Load the transformer model
transformer = TFLongformerForSequenceClassification.from_pretrained('allenai/longformer-base-4096', return_dict=True)

# Custom layer to handle transformer
class TransformerLayer(Layer):
    def __init__(self, model):
        super(TransformerLayer, self).__init__()
        self.model = model

    @tf.function
    def call(self, inputs):
        input_ids, attention_mask = inputs
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

# Assuming the Transformer outputs (batch_size, sequence_length, num_labels) and you are using the CLS token
transformer_output = TransformerLayer(transformer)([input_ids, attention_mask])
output = Dense(3, activation='softmax')(transformer_output)

import tensorflow as tf

class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.round(y_pred)
        y_true = tf.cast(y_true, tf.float32)  # Ensure y_true is float32
        y_pred = tf.cast(y_pred, tf.float32)  # Ensure y_pred is float32

        true_positives = tf.reduce_sum(y_true * y_pred)
        false_positives = tf.reduce_sum((1 - y_true) * y_pred)
        false_negatives = tf.reduce_sum(y_true * (1 - y_pred))

        self.true_positives.assign_add(true_positives)
        self.false_positives.assign_add(false_positives)
        self.false_negatives.assign_add(false_negatives)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))

    def reset_states(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)


# Build and compile the model
model = Model(inputs=[input_ids, attention_mask], outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', F1Score()])

# Prepare datasets for training
train_dataset = tf.data.Dataset.from_tensor_slices(({
    "input_ids": X_train_enc['input_ids'],
    "attention_mask": X_train_enc['attention_mask']
}, y_train)).batch(4)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    "input_ids": X_test_enc['input_ids'],
    "attention_mask": X_test_enc['attention_mask']
}, y_test)).batch(4)

# Train the model
model.fit(train_dataset, epochs=20, validation_data=test_dataset)


c:\Users\jpers\Desktop\NLP\transformers_env\Lib\site-packages\tf_keras\src\initializers\initializers.py:121: UserWarning: The initializer TruncatedNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(
Some layers from the model checkpoint at allenai/longformer-base-4096 were not used when initializing TFLongformerForSequenceClassification: ['lm_head']
- This IS expected if you are initializing TFLongformerForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFLongformerForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializin

Epoch 1/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2362s 7s/step - accuracy: 0.5889 - f1_score: 0.0000e+00 - loss: 1.0325 - val_accuracy: 0.7934 - val_f1_score: 0.0000e+00 - val_loss: 0.8524
Epoch 2/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2386s 7s/step - accuracy: 0.7205 - f1_score: 0.3769 - loss: 0.8601 - val_accuracy: 0.7934 - val_f1_score: 0.7934 - val_loss: 0.7305
Epoch 3/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2365s 7s/step - accuracy: 0.7205 - f1_score: 0.7205 - loss: 0.7813 - val_accuracy: 0.7934 - val_f1_score: 0.7934 - val_loss: 0.6712
Epoch 4/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2367s 7s/step - accuracy: 0.7205 - f1_score: 0.7205 - loss: 0.7463 - val_accuracy: 0.7934 - val_f1_score: 0.7934 - val_loss: 0.6415
Epoch 5/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2377s 7s/step - accuracy: 0.7205 - f1_score: 0.7205 - loss: 0.7299 - val_accuracy: 0.7934 - val_f1_score: 0.7934 - val_loss: 0.6256
Epoch 6/20
334/334 ━━━━━━━━━━━━━━━━━━━━ 2360s 7s/step - accuracy: 0.7205 - f1_score: 0.7205 - loss: 0.7215 - val_accuracy: 0.7934 - 